# Pulling Data in at the ZCTA level for our Restaurant Health Inspection Scores Dataset

Notebook opened: 4/13/25

In [1]:
#imports
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

Links to Census data:
- 2020 Census ZCTAs https://www.census.gov/programs-surveys/geography/guidance/geo-areas/zctas.html


In [2]:
#loading data

#geocoded restaurant health inspection data
df = pd.read_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/healthinspections_2024_geocoded.csv")

/var/folders/81/w_61xz297rv4ggdktb58tlxm0000gn/T/ipykernel_35824/2128334811.py:4: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/healthinspections_2024_geocoded.csv")


In [3]:
#convert dataframe into GeoDataFrame
print(df.columns.tolist())

#Creating geometry column
geometry = gpd.points_from_xy(df.Longitude, df.Latitude)
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")  # WGS84


['INSPECTION_DATE', 'STORE_NAME', 'STREET_ADDRESS', 'CITY', 'ZIP5', 'SERVICE_DESCRIPTION', 'SCORE', 'GRADE', 'STREET_ADDRESS_LINE2', 'INSPDATE_YEAR', 'STATE', 'Latitude', 'Longitude', 'Accuracy Score', 'Accuracy Type', 'Number', 'Street', 'Unit Type', 'Unit Number', 'City', 'State', 'County', 'Zip', 'Country', 'Source']


In [4]:
# Reading in ZCTA boundaries
zcta_gdf = gpd.read_file("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Raw Data/zcta_shapefiles/tl_2020_us_zcta520.shp").to_crs("EPSG:3857") 
# projecting ZCTA shapefile to a metric CRS (for distance calculation)
zcta_gdf = zcta_gdf.to_crs("EPSG:3857")  # Web Mercator: units in meters
restaurants_gdf = gdf.to_crs("EPSG:3857")


In [5]:
#doing a spatial join between restaurants df and zcta df with geometry
restaurants_with_zcta = gpd.sjoin(restaurants_gdf, zcta_gdf[['ZCTA5CE20', 'geometry']], how="left", predicate="within")
restaurants_with_zcta = restaurants_with_zcta.rename(columns={"ZCTA5CE20": "PRIMARY_ZCTA"})

In [6]:
grouped_by_zcta = restaurants_with_zcta.groupby('PRIMARY_ZCTA').size()

# Print the value counts for each ZCTA
print(grouped_by_zcta)

PRIMARY_ZCTA
40023      2
40041     10
40059    105
40118     38
40177      1
        ... 
93551    193
93552     79
93553      3
93563      3
93591      9
Length: 378, dtype: int64


In [7]:
#downloading census data
import censusdis.data as ced

#Variables we want to query
# Variables = ["MEDIAN_HOUSEHOLD_INCOME_VARIABLE", "AVG_HOUSEHOLD_SIZE", "TOTAL_POPULATION", "MEDIAN GROSS RENT"]
VARIABLES = ["NAME","B19013_001E", "B25010_001E", "B01003_001E", "B25064_001E"]

ACS2023_indicators = ced.download(
    dataset="acs/acs5",
    vintage=2023,
    download_variables=VARIABLES,
    zip_code_tabulation_area="*",
    with_geometry=True
)

In [8]:
#merging into zcta_gdf
ACS2023_indicators = ACS2023_indicators.rename(columns={"B19013_001E": "MEDIAN_INCOME", "B25010_001E": "AVG_HOUSEHOLD_SIZE", "B01003_001E": "TOTAL_POPULATION", "B25064_001E": "MEDIAN_GROSS_RENT", "ZIP_CODE_TABULATION_AREA": "ZCTA"})
zcta_gdf = zcta_gdf.merge(ACS2023_indicators[['ZCTA', 'MEDIAN_INCOME', 'AVG_HOUSEHOLD_SIZE', 'TOTAL_POPULATION', 'MEDIAN_GROSS_RENT']], left_on="ZCTA5CE20", right_on="ZCTA", how="left")

In [9]:
#For each restaurant, find nearby ZCTAs within 2 miles and average
from shapely.geometry import Point

# Remove any existing 'index_right' column to prevent conflict
if 'index_right' in restaurants_with_zcta.columns:
    restaurants_with_zcta = restaurants_with_zcta.drop(columns='index_right')

# Convert 2 miles to meters
buffer_radius = 3218.69  # meters

# Create buffer around each restaurant point
restaurants_with_zcta['buffer'] = restaurants_with_zcta.geometry.buffer(buffer_radius)

# Use spatial join: buffered restaurant areas to intersecting ZCTAs
buffered_join = gpd.sjoin(
    gpd.GeoDataFrame(restaurants_with_zcta, geometry='buffer'),
    zcta_gdf[['ZCTA5CE20', 'geometry', 'MEDIAN_INCOME', 'AVG_HOUSEHOLD_SIZE', 'TOTAL_POPULATION', 'MEDIAN_GROSS_RENT']],
    how='left',
    predicate='intersects'
)

# Group by restaurant and calculate average income of surrounding ZCTAs
avg_censusindicators_nearby = buffered_join.groupby(buffered_join.index).agg(
    AVG_INCOME_NEARBY_ZCTAS=('MEDIAN_INCOME', 'mean'),
    AVG_HH_SIZE_NEARBY_ZCTAS=('AVG_HOUSEHOLD_SIZE', 'mean'),
    AVG_RENT_NEARBY_ZCTAS=('MEDIAN_GROSS_RENT', 'mean'),
    AVG_POP_NEARBY_ZCTAS=('TOTAL_POPULATION', 'mean')
)

# Merge the average income back into the original restaurant dataframe
restaurants_final = restaurants_with_zcta.join(avg_censusindicators_nearby)


In [10]:
restaurants_final.head()

,INSPECTION_DATE,STORE_NAME,STREET_ADDRESS,CITY,ZIP5,SERVICE_DESCRIPTION,SCORE,GRADE,STREET_ADDRESS_LINE2,INSPDATE_YEAR,...,Zip,Country,Source,geometry,PRIMARY_ZCTA,buffer,AVG_INCOME_NEARBY_ZCTAS,AVG_HH_SIZE_NEARBY_ZCTAS,AVG_RENT_NEARBY_ZCTAS,AVG_POP_NEARBY_ZCTAS
0,2024-01-02,3RD ST CHEVRON,3817 W 3RD ST,LOS ANGELES,90020,ROUTINE INSPECTION,90.0,A,NaN,2024,...,90020.0,US,City of Los Angeles (CC0 1.0),POINT (-13168932.121 4038094.997),90004,"POLYGON ((-13165713.431 4038094.997, -13165728...",64645.500000,2.320000,1755.800000,42499.700000
1,2024-01-02,7-ELEVEN #34473B,5905 SOUTH ST,LAKEWOOD,90713,ROUTINE INSPECTION,97.0,A,NaN,2024,...,90713.0,US,Los Angeles (Los Angeles County license to cop...,POINT (-13148707.93 4009888.065),90713,"POLYGON ((-13145489.24 4009888.065, -13145504....",112001.714286,3.130000,2346.428571,37331.571429
2,2024-01-02,7-ELEVEN #39023B,4111 VENICE BLVD,LOS ANGELES,90019,ROUTINE INSPECTION,96.0,A,NaN,2024,...,90019.0,US,City of Los Angeles (CC0 1.0),POINT (-13172079.123 4034756.016),90019,"POLYGON ((-13168860.433 4034756.016, -13168875...",67153.000000,2.385556,1848.000000,41001.888889
3,2024-01-02,7-ELEVEN #39714,6051 HOLLYWOOD BLVD # 111,LOS ANGELES,90028,ROUTINE INSPECTION,94.0,A,NaN,2024,...,90028.0,US,City of Los Angeles (CC0 1.0),POINT (-13171547.016 4042482.172),90028,"POLYGON ((-13168328.326 4042482.172, -13168343...",79224.375000,1.988750,1992.750000,38403.625000
4,2024-01-02,AEIRLOOM-CATCHER,10550 RIVERSIDE DR,NORTH HOLLYWOOD,91602,ROUTINE INSPECTION,92.0,A,NaN,2024,...,91602.0,US,City of Los Angeles (CC0 1.0),POINT (-13175841.722 4049232.909),91602,"POLYGON ((-13172623.032 4049232.909, -13172638...",100588.428571,1.932500,2194.714286,27773.000000


In [ ]:
#Adding Social Vulnerability Index Variables


In [11]:
#Export
output_path = "/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/healthinspections_2024_censusindicators.csv"
restaurants_final.to_csv(output_path, index=False)